In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.stores as
select *, 
  current_timestamp() as ingested_at,
  _metadata.file_path as _source_file
  from read_files("/Volumes/{catalog}/landing/raw/stores/stores.csv")
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.customers
""")

spark.sql(f"""
COPY INTO {catalog}.bronze.customers
FROM "/Volumes/{catalog}/landing/raw/customers/"
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint, num_skipped_corrupt_files: int]

In [0]:
%sql
drop table if exists fk_ce1366.bronze.products

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.products
""")

spark.sql(f"""
COPY INTO {catalog}.bronze.products
FROM (select *, _metadata.file_path as _source_file,
        current_timestamp() as ingested_at
      from"/Volumes/{catalog}/landing/raw/products/")
FILEFORMAT = JSON
FORMAT_OPTIONS ("multiline" = "true")
COPY_OPTIONS ('mergeSchema' = 'true')
""")



DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint, num_skipped_corrupt_files: int]

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/ops/pipeline_state/order_schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudfiles.inferColumnTypes", "true")
    .option("multiline", "true")
    .load(f"/Volumes/{catalog}/landing/raw/orders/")
    .select(
            "*",
            col("_metadata.file_path").alias("_source_file"),
            current_timestamp().alias("ingested_at")
        )
    .writeStream
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/orders_checkpoint")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.orders")
  )


In [0]:
%sql
select * from fk_ce1366.bronze.orders


customer_id,delivery,items,order_id,order_status,order_ts,payment,store_id,updated_at,_rescued_data,_source_file,ingested_at
C0383,"List(21, 0.0, 15)","List(List(0.0, P006, 3, 35.0), List(63.0, P044, 4, 85.0), List(26.0, P002, 1, 159.0), List(0.0, P034, 1, 20.0))",FK-20260304-0126,DELIVERED,2026-03-04 09:39:57,"List(Card, TXN3599496630)",S10,2026-03-04 10:00:57,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0080,"List(8, 25.0, 10)","List(List(null, P041, 1, 132.0))",FK-20260304-0146,Delivered,2026-03-04 21:27:55,"List(UPI, TXN7728817559)",S03,2026-03-04 21:35:55,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0238,"List(15, 25.0, 15)","List(List(0.0, P014, 1, 42.0), List(null, P011, 3, 28.0), List(null, P008, 1, 10.0))",FK-20260304-0077,DELIVERED,2026-03-04 18:01:48,"List( Upi, TXN4626827068)",S08,2026-03-04 18:16:48,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0264,"List(14, 0.0, 15)","List(List(null, P024, 1, 105.0), List(0.0, P021, 1, 279.0), List(11.0, P052, 1, 79.0), List(null, P041, 1, 132.0), List(null, P003, 2, 135.0))",FK-20260303-0107,Delivered,2026-03-03 21:54:45,"List(CARD, TXN3689645593)",S12,2026-03-04 21:58:45,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0148,"List(9, 0.0, 10)","List(List(null, P006, 1, 35.0), List(null, P022, 4, 265.0), List(9.0, P049, 1, 128.0))",FK-20260304-0025,Delivered,2026-03-04 18:13:55,"List( Upi, TXN5185134165)",S07,2026-03-04 18:22:55,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0094,"List(14, 0.0, 15)","List(List(46.0, P039, 1, 259.0), List(0.0, P014, 2, 42.0))",FK-20260304-0078,delivered,2026-03-04 12:15:00,"List(upi, TXN2075593066)",S08,2026-03-04 12:29:00,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0324,"List(5, 0.0, 10)","List(List(24.0, P049, 2, 128.0), List(null, P037, 1, 38.0))",FK-20260304-0042,DELIVERED,2026-03-04 23:12:32,"List(UPI, TXN1834876774)",S11,2026-03-04 23:17:32,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0169,"List(7, 0.0, 10)","List(List(null, P022, 4, 265.0), List(null, P014, 2, 42.0), List(25.0, P028, 1, 149.0), List(0.0, P004, 1, 39.0), List(null, P027, 2, 79.0))",FK-20260304-0167,Delivered,2026-03-04 16:39:26,"List(UPI, TXN9477764723)",S01,2026-03-04 16:46:26,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0164,"List(5, 25.0, 10)","List(List(null, P001, 1, 54.0))",FK-20260304-0147,DELIVERED,2026-03-04 21:05:20,"List(upi, TXN5362969705)",S03,2026-03-04 21:10:20,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z
C0196,"List(null, 0.0, 10)","List(List(0.0, P039, 4, 259.0), List(0.0, P015, 3, 54.0))",FK-20260304-0022,placed,2026-03-04 15:46:09,"List(UPI, TXN8437492385)",S07,2026-03-04 15:47:09,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z


In [0]:
%sql
select * from fk_ce1366.bronze.products

brand,category,is_active,mrp,pack,product_id,product_name,selling_price,snapshot_date,tags,_rescued_data,_source_file,ingested_at
FreshKart Farms,"List(Fruits & Vegetables, Fresh Fruits)",true,60.0,"List(6, pcs)",P001,Banana Robusta,54.0,2026-03-01,"List(fresh, daily)",null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Fruits)",true,180.0,"List(4, pcs)",P002,Apple Shimla,159.0,2026-03-01,List(fresh),null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Fruits)",true,150.0,"List(3, pcs)",P003,Pomegranate,135.0,2026-03-01,List(fresh),null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,45.0,"List(1, kg)",P004,Onion,39.0,2026-03-01,"List(fresh, daily, staple)",null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,30.0,"List(500, g)",P005,Tomato Hybrid,26.0,2026-03-01,"List(fresh, daily)",null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,40.0,"List(1, kg)",P006,Potato,35.0,2026-03-01,"List(fresh, staple)",null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,15.0,"List(100, g)",P007,Coriander Leaves,12.0,2026-03-01,"List(fresh, daily)",null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,12.0,"List(100, g)",P008,Green Chilli,10.0,2026-03-01,List(fresh),null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,30.0,"List(250, g)",P009,Lemon,25.0,2026-03-01,List(fresh),null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z
FreshKart Farms,"List(Fruits & Vegetables, Fresh Vegetables)",true,28.0,"List(500, g)",P010,Cucumber,24.0,2026-03-01,List(fresh),null,/Volumes/fk_ce1366/landing/raw/products/products_2026-03-01.json,2026-10-12T11:50:37.501Z


In [0]:
%sql
select * from fk_ce1366.bronze.customers

customer_id,full_name,email,phone,city,tier,signup_date
C0001,Kiara Mehta,kiara.mehta1@outlook.com,9967103366,Jaipur,BASIC,20/06/2024
C0002,Aditya Gupta,aditya.gupta2@gmail.com,+91-98398-14388,Delhi,Premium,2024-01-26
C0003,Ananya Shekhawat,ANANYA.SHEKHAWAT3@YAHOO.IN,919935231776,Mumbai,PLUS,19/10/2024
C0004,Yash Iyer,yash.iyer4@gmail.com,9014941883,Bengaluru,Premium,2025-05-27
C0005,Tara Nair,tara.nair5@outlook.com,9445672481,Pune,Plus,2023-03-10
C0006,Diya Joshi,diya.joshi6@yahoo.in,9236469229,Hyderabad,BASIC,19/11/2024
C0007,Nikhil Nair,nikhil.nair7@yahoo.in,9817804830,JAIPUR,Plus,16/04/2025
C0008,Varun Gupta,varun.gupta8@yahoo.in,099490 18941,Delhi,BASIC,2025-08-07
C0009,Kiara Gupta,kiara.gupta9@outlook.com,9064120097,Mumbai,BASIC,2023-10-03
C0010,Rahul Malhotra,rahul.malhotra10@outlook.com,+91 9564035207,Bengaluru,Basic,30/11/2024


In [0]:
%sql
select * from fk_ce1366.bronze.stores

store_id,store_name,city,state,region,opening_date,_rescued_data,ingested_at,_source_file
S01,FreshKart Malviya Nagar,jaipur,Rajasthan,North,2024-06-15,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S02,FreshKart Vaishali Nagar,Jaipur,Rajasthan,North,2024-09-01,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S03,FreshKart Saket,Delhi,Delhi,North,2023-11-20,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S04,FreshKart Dwarka,DELHI,Delhi,North,2024-02-10,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S05,FreshKart Andheri West,Mumbai,Maharashtra,West,2023-08-05,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S06,FreshKart Powai,mumbai,Maharashtra,West,2024-01-18,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S07,FreshKart Koramangala,Bengaluru,Karnataka,South,2023-07-01,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S08,FreshKart Whitefield,Bengaluru,Karnataka,South,2024-04-22,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S09,FreshKart Kothrud,Pune,Maharashtra,West,2024-05-30,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
S10,FreshKart Hinjewadi,PUNE,Maharashtra,West,2024-10-12,null,2026-10-12T08:08:16.506Z,dbfs:/Volumes/fk_ce1366/landing/raw/stores/stores.csv
